# Generate wide Salesforce Account-style CSVs

Creates objects under **your** Unity Catalog (`<catalog>.<schema>`), writes CSV exports to a **volume**, and writes a **manifest** JSON that records each file's column order (including intentional header quirks).

When you run the included **Databricks Asset Bundle** job, `catalog`, `schema`, and `volume_name` are supplied from `databricks.yml` variables. When you run this notebook alone, set the same values in the widgets (or leave blank to use the defaults in code).

For a first run, keep `ROWS_PER_FILE` modest; increase it after the load notebook succeeds.


## Configuration

Set **catalog**, **schema**, and **volume** via the job (bundle) or widgets. Defaults apply when a widget is left empty.


In [ ]:
# Unity Catalog: from job parameters (bundle) or widgets (interactive). No spark.conf.set for app config.
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("volume_name", "", "Volume name")


def _param(name: str, default: str) -> str:
    try:
        v = dbutils.widgets.get(name).strip()
        return v if v else default
    except Exception:
        return default


CATALOG = _param("catalog", "main")
SCHEMA = _param("schema", "sf_account_csv_lab")
VOLUME_NAME = _param("volume_name", "sf_account_raw")

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
CSV_SUBDIR = "csv"
MANIFEST_SUBDIR = "manifest"
MANIFEST_NAME = "file_manifest.json"

ROWS_PER_FILE = 10_000
NUM_FILES = 50
BATCH_SIZE = 4
SEED = 42

_STD_ACCOUNT_FIELDS = (
    "Id Name Type ParentId AccountNumber Site AccountSource AnnualRevenue NumberOfEmployees "
    "Industry Ownership TickerSymbol Rating BillingStreet BillingCity BillingState BillingPostalCode BillingCountry "
    "BillingLatitude BillingLongitude ShippingStreet ShippingCity ShippingState ShippingPostalCode ShippingCountry "
    "Phone Fax Website Description OwnerId CreatedDate LastModifiedDate LastActivityDate IsDeleted PhotoUrl "
    "CleanStatus DunsNumber NaicsCode NaicsDesc SicDesc"
).split()
_CUSTOM_FIELDS = [f"Segment_Score_{i:03d}__c" for i in range(1, 461)]
CANONICAL_COLUMNS = _STD_ACCOUNT_FIELDS + _CUSTOM_FIELDS

assert len(CANONICAL_COLUMNS) == 500

print("BASE_PATH:", BASE_PATH)
print("ROWS_PER_FILE:", ROWS_PER_FILE, "NUM_FILES:", NUM_FILES)


## Create schema and volume

Run once per workspace; safe to re-run.


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"OK: {CATALOG}.{SCHEMA} volume {VOLUME_NAME}")


## Prepare directories

Ensures `csv/` and `manifest/` exist under the volume.


In [ ]:
for sub in (CSV_SUBDIR, MANIFEST_SUBDIR):
    dbutils.fs.mkdirs(f"{BASE_PATH}/{sub}")
print("Directories ready")


## Generate CSV files and manifest

For each file index, computes a **batch** (every `BATCH_SIZE` files), derives **column count** between 400 and 500, takes a **prefix** of the canonical column list (simulates appended fields over time), **shuffles** order per file, and writes **one** CSV per file via a temporary Parquet/CSV staging path.

Values use **picklists, addresses, phones, dates, and custom `Segment_Score_*__c` metrics** derived per row from hashes (repeatable for a given export). **Lineage:** `Id` is Salesforce-style `001` + 15 hex chars (unique per row and file batch).


In [ ]:
import json
import random
from pyspark.sql import functions as F

N_BATCHES = (NUM_FILES - 1) // BATCH_SIZE + 1


def column_count_for_batch(batch_idx: int) -> int:
    if N_BATCHES <= 1:
        return min(500, len(CANONICAL_COLUMNS))
    frac = batch_idx / max(1, (N_BATCHES - 1))
    n = 400 + int(round((500 - 400) * frac))
    return int(min(500, max(400, n)))


def _arr_pick(values, hb):
    arr = F.array(*[F.lit(v) for v in values])
    n = len(values)
    idx = (F.pmod(hb, F.lit(n).cast("bigint")) + F.lit(1).cast("bigint")).cast("int")  # pmod is non-negative; no abs (abs(MIN_INT) overflows ANSI)
    return F.element_at(arr, idx)


def account_column(name: str, rid, file_idx: int) -> F.Column:
    # bigint hash; use pmod(...) not abs(hash) — abs(Integer.MIN_VALUE) raises ARITHMETIC_OVERFLOW under ANSI
    hb = F.hash(rid, F.lit(file_idx), F.lit(name)).cast("bigint")
    hs = F.sha2(F.concat(rid.cast("string"), F.lit("|"), F.lit(str(file_idx)), F.lit("|"), F.lit(name)), 256)

    if name == "Id":
        return F.concat(F.lit("001"), F.upper(F.substring(hs, 1, 15)))
    if name == "Name":
        p1 = _arr_pick(
            ["Acme Corp", "Globex LLC", "Initech", "Umbrella Holdings", "Wayne Enterprises", "Stark Ind", "Wonka Foods", "Hooli", "Vehement Capital"],
            hb,
        )
        p2 = _arr_pick(
            ["Manufacturing", "Logistics", "Healthcare", "Retail", "Automotive", "Aerospace", "CPG", "MedTech"],
            hb,
        )
        return F.concat(p1, F.lit(" — "), p2, F.lit(" (#"), F.lpad(rid.cast("string"), 6, "0"), F.lit(")"))
    if name == "Type":
        return _arr_pick(["Customer", "Partner", "Prospect", "Other", "Competitor", "Reseller", "Integrator"], hb)
    if name == "ParentId":
        pid = F.concat(F.lit("001"), F.upper(F.substring(F.sha2(F.concat(rid.cast("string"), F.lit("PARENT")), 256), 1, 15)))
        return F.when(F.pmod(hb, F.lit(4).cast("bigint")) == F.lit(0).cast("bigint"), F.lit(None).cast("string")).otherwise(pid)
    if name == "AccountNumber":
        return F.concat(
            F.lit("AN-"),
            F.lpad((F.pmod(hb, F.lit(90_000_000).cast("bigint")) + F.lit(10_000_000).cast("bigint")).cast("string"), 8, "0"),
        )
    if name == "Site":
        return _arr_pick(["HQ", "Plant 01", "Plant 02", "DC-East", "DC-West", "Regional Office"], hb)
    if name == "AccountSource":
        return _arr_pick(
            ["Web", "Partner Referral", "Purchased List", "Trade Show", "Employee Referral", "Phone Inquiry", "Event"],
            hb,
        )
    if name == "AnnualRevenue":
        rev = (F.pmod(hb, F.lit(250_000_000).cast("bigint")) + F.lit(250_000).cast("bigint")).cast("string")
        return F.concat(rev, F.lit(".00"))
    if name == "NumberOfEmployees":
        return (F.pmod(hb, F.lit(48_000).cast("bigint")) + F.lit(12).cast("bigint")).cast("string")
    if name == "Industry":
        return _arr_pick(
            [
                "Manufacturing",
                "Technology",
                "Healthcare",
                "Retail",
                "Finance",
                "Energy",
                "Transportation",
                "Construction",
                "Agriculture",
                "Public Sector",
            ],
            hb,
        )
    if name == "Ownership":
        return _arr_pick(["Public", "Private", "Subsidiary", "Government", "Nonprofit"], hb)
    if name == "TickerSymbol":
        return F.upper(F.substring(hs, 3, 4))
    if name == "Rating":
        return _arr_pick(["Hot", "Warm", "Cold", "Not Rated"], hb)
    if name == "BillingStreet":
        num = (F.pmod(hb, F.lit(9999).cast("bigint")) + F.lit(1).cast("bigint")).cast("string")
        street = _arr_pick(["Main St", "Oak Ave", "Industrial Pkwy", "Commerce Dr", "River Rd", "Maple Ln"], hb)
        return F.concat(num, F.lit(" "), street)
    if name == "BillingCity":
        return _arr_pick(
            ["Chicago", "Dallas", "Phoenix", "Atlanta", "Denver", "Seattle", "Detroit", "Boston", "Minneapolis", "Tampa"],
            hb,
        )
    if name == "BillingState":
        return _arr_pick(["IL", "TX", "AZ", "GA", "CO", "WA", "MI", "MA", "MN", "FL", "CA", "OH", "PA", "NY"], hb)
    if name == "BillingPostalCode":
        return F.concat(
            F.lpad((F.pmod(hb, F.lit(90000).cast("bigint")) + F.lit(10000).cast("bigint")).cast("string"), 5, "0")
        )
    if name == "BillingCountry":
        return _arr_pick(["United States", "Canada", "Mexico", "United Kingdom", "Germany"], hb)
    if name == "BillingLatitude":
        deg = F.lpad((F.pmod(hb, F.lit(89).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        frac = F.lpad(F.pmod(hb, F.lit(99999).cast("bigint")).cast("string"), 5, "0")
        return F.concat(
            F.when(F.pmod(hb, F.lit(2).cast("bigint")) == F.lit(0).cast("bigint"), F.lit("-")).otherwise(F.lit("")),
            deg,
            F.lit("."),
            frac,
        )
    if name == "BillingLongitude":
        deg = F.lpad((F.pmod(hb, F.lit(179).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 3, "0")
        frac = F.lpad(F.pmod(hb, F.lit(99999).cast("bigint")).cast("string"), 5, "0")
        return F.concat(
            F.when(F.pmod(hb, F.lit(2).cast("bigint")) == F.lit(0).cast("bigint"), F.lit("-")).otherwise(F.lit("")),
            deg,
            F.lit("."),
            frac,
        )
    if name == "ShippingStreet":
        return F.concat(F.lit("Ship: "), account_column("BillingStreet", rid, file_idx))
    if name == "ShippingCity":
        return account_column("BillingCity", rid, file_idx)
    if name == "ShippingState":
        return account_column("BillingState", rid, file_idx)
    if name == "ShippingPostalCode":
        return account_column("BillingPostalCode", rid, file_idx)
    if name == "ShippingCountry":
        return account_column("BillingCountry", rid, file_idx)
    if name == "Phone":
        area = F.lpad((F.pmod(hb, F.lit(799).cast("bigint")) + F.lit(200).cast("bigint")).cast("string"), 3, "0")
        mid = F.lpad((F.pmod(hb, F.lit(799).cast("bigint")) + F.lit(200).cast("bigint")).cast("string"), 3, "0")
        last = F.lpad((F.pmod(hb, F.lit(9000).cast("bigint")) + F.lit(1000).cast("bigint")).cast("string"), 4, "0")
        return F.concat(F.lit("+1-"), area, F.lit("-"), mid, F.lit("-"), last)
    if name == "Fax":
        return F.when(
            F.pmod(hb, F.lit(2).cast("bigint")) == F.lit(0).cast("bigint"),
            F.lit(None).cast("string"),
        ).otherwise(account_column("Phone", rid, file_idx))
    if name == "Website":
        slug = F.lower(F.concat(F.substring(hs, 1, 6), F.lit("-"), F.lpad(rid.cast("string"), 4, "0")))
        return F.concat(F.lit("https://www."), slug, F.lit(".example.com"))
    if name == "Description":
        seg1 = _arr_pick(
            ["Strategic account", "National distributor", "Regional provider", "Key OEM partner", "Growth target"],
            hb,
        )
        seg2 = _arr_pick(
            ["focus on aftermarket", "multi-site rollout", "ERP migration in flight", "expanding capacity"],
            hb,
        )
        return F.concat(seg1, F.lit("; "), seg2, F.lit("."))
    if name == "OwnerId":
        return F.concat(F.lit("005"), F.upper(F.substring(hs, 5, 12)))
    if name == "CreatedDate":
        y = (F.lit(2018).cast("bigint") + F.pmod(hb, F.lit(7).cast("bigint"))).cast("string")
        mo = F.lpad((F.pmod(hb, F.lit(12).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        d = F.lpad((F.pmod(hb, F.lit(28).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        return F.concat(y, F.lit("-"), mo, F.lit("-"), d)
    if name == "LastModifiedDate":
        y = (F.lit(2019).cast("bigint") + F.pmod(hb, F.lit(6).cast("bigint"))).cast("string")
        mo = F.lpad((F.pmod(hb, F.lit(12).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        d = F.lpad((F.pmod(hb, F.lit(28).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        hh = F.lpad(F.pmod(hb, F.lit(24).cast("bigint")).cast("string"), 2, "0")
        mi = F.lpad(F.pmod(hb, F.lit(60).cast("bigint")).cast("string"), 2, "0")
        return F.concat(y, F.lit("-"), mo, F.lit("-"), d, F.lit("T"), hh, F.lit(":"), mi, F.lit(":00.000Z"))
    if name == "LastActivityDate":
        y = (F.lit(2020).cast("bigint") + F.pmod(hb, F.lit(5).cast("bigint"))).cast("string")
        mo = F.lpad((F.pmod(hb, F.lit(12).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        d = F.lpad((F.pmod(hb, F.lit(28).cast("bigint")) + F.lit(1).cast("bigint")).cast("string"), 2, "0")
        return F.when(
            F.pmod(hb, F.lit(5).cast("bigint")) == F.lit(0).cast("bigint"),
            F.lit(None).cast("string"),
        ).otherwise(F.concat(y, F.lit("-"), mo, F.lit("-"), d))
    if name == "IsDeleted":
        return F.when(F.pmod(hb, F.lit(400).cast("bigint")) == F.lit(0).cast("bigint"), F.lit("true")).otherwise(F.lit("false"))
    if name == "PhotoUrl":
        return F.concat(F.lit("/services/images/photo/"), F.substring(hs, 1, 18))
    if name == "CleanStatus":
        return _arr_pick(["Pending", "In Sync", "Different", "Not Found", "Skipped"], hb)
    if name == "DunsNumber":
        return F.lpad(
            (F.pmod(hb, F.lit(900_000_000).cast("bigint")) + F.lit(100_000_000).cast("bigint")).cast("string"),
            9,
            "0",
        )
    if name == "NaicsCode":
        return F.lpad(
            (F.pmod(hb, F.lit(899_999).cast("bigint")) + F.lit(100_000).cast("bigint")).cast("string"),
            6,
            "0",
        )
    if name == "NaicsDesc":
        return _arr_pick(
            ["Medical equipment mfg", "Software publishers", "Freight trucking", "General warehousing", "Surgical hospitals"],
            hb,
        )
    if name == "SicDesc":
        return _arr_pick(["Electronic components", "Industrial machinery", "Grocery stores", "Insurance carriers"], hb)
    if name.startswith("Segment_Score_") and name.endswith("__c"):
        whole = (F.pmod(hb, F.lit(999_000).cast("bigint")) + F.lit(1_000).cast("bigint")).cast("string")
        frac = F.lpad(F.pmod(hb, F.lit(100).cast("bigint")).cast("string"), 2, "0")
        return F.concat(whole, F.lit("."), frac)
    return F.concat(F.lit(name), F.lit("="), rid.cast("string"))


manifest = []

for file_idx in range(1, NUM_FILES + 1):
    batch_idx = (file_idx - 1) // BATCH_SIZE
    ncols = column_count_for_batch(batch_idx)
    col_subset = CANONICAL_COLUMNS[:ncols]
    rng = random.Random(SEED + file_idx * 1_000 + batch_idx)
    ordered = list(col_subset)
    rng.shuffle(ordered)

    rid = F.col("id")
    expr_map = {c: account_column(c, rid, file_idx) for c in col_subset}

    select_list = [expr_map[c].alias(c) for c in ordered]
    df_out = spark.range(ROWS_PER_FILE).select(*select_list)

    # Controlled anomalies (loader must tolerate)
    if file_idx % 11 == 0:
        # Ragged rows: null out two logical columns for ~5% of rows (trailing fields empty in export)
        victim_cols = [c for c in ordered if c not in ("Id", "Name")][:2]
        if len(victim_cols) == 2:
            c_a, c_b = victim_cols
            cond = F.pmod(F.hash(F.col("Id")).cast("bigint"), F.lit(20).cast("bigint")) == F.lit(0).cast("bigint")
            df_out = df_out.withColumn(
                c_a, F.when(cond, F.lit(None).cast("string")).otherwise(F.col(c_a))
            ).withColumn(
                c_b, F.when(cond, F.lit(None).cast("string")).otherwise(F.col(c_b))
            )

    if file_idx % 17 == 0:
        # Blank-style header placeholders (still unique Spark names; manifest records display names)
        rename_map = {}
        for j, c in enumerate(ordered):
            if c not in ("Id", "Name") and j % 97 == 5:
                rename_map[c] = f"_BLANK_HDR_{j}"
        for old, new in rename_map.items():
            df_out = df_out.withColumnRenamed(old, new)
        ordered_display = [rename_map.get(c, c) for c in ordered]
    else:
        ordered_display = list(ordered)

    staging = f"{BASE_PATH}/_tmp_csv_{file_idx:03d}"
    out_csv = f"{BASE_PATH}/{CSV_SUBDIR}/accounts_{file_idx:03d}.csv"
    dbutils.fs.rm(staging, True)
    df_out.coalesce(1).write.mode("overwrite").option("header", True).option("delimiter", ",").csv(staging)

    parts = [
        x.path
        for x in dbutils.fs.ls(staging)
        if x.name.startswith("part-") and x.name.endswith(".csv")
    ]
    if not parts:
        raise RuntimeError(f"No part csv under {staging}")
    dbutils.fs.cp(parts[0], out_csv, True)
    dbutils.fs.rm(staging, True)

    manifest.append(
        {
            "file_index": file_idx,
            "volume_path": out_csv,
            "columns": ordered_display,
            "logical_columns": ordered,
            "column_count": len(ordered_display),
            "rows": ROWS_PER_FILE,
            "batch": batch_idx,
            "anomaly_ragged_nulls": bool(file_idx % 11 == 0),
            "anomaly_blank_like_headers": bool(file_idx % 17 == 0),
        }
    )

# Compact JSON (no indent) — smaller file; indented manifest can exceed dbutils.fs.head 1 MiB default in verify
man_body = json.dumps(manifest, separators=(",", ":"))
man_target = f"{BASE_PATH}/{MANIFEST_SUBDIR}/{MANIFEST_NAME}"
dbutils.fs.put(man_target, man_body, True)
print(f"Wrote manifest: {man_target}")
print(f"CSV files: {NUM_FILES} under {BASE_PATH}/{CSV_SUBDIR}/")


## Quick verification

Reads the **entire** manifest with `dbutils.fs.open` (not `head`, which is limited to ~1 MiB), then parses JSON.


In [ ]:
# dbutils.fs.head() is capped at 1 MiB in many runtimes — read the full file for json.loads
_stream = dbutils.fs.open(man_target, "rb")
try:
    _raw = _stream.read().decode("utf-8")
finally:
    _stream.close()
m = json.loads(_raw)
print("manifest entries:", len(m))
print("sample:", m[0]["volume_path"], "cols:", m[0]["column_count"])
